In [3]:
import pygrib
import math
import calendar
import pandas as pd


In [20]:
def decodeGrib(grb,data,latlons):
    dat=grb.data()
    adate=f"{str(grb.analDate.date().month).zfill(2):2s}/{str(grb.analDate.date().day).zfill(2)}/{str(grb.analDate.date().year).zfill(2)}"
    nfct=grb.forecastTime
    fdate = f"{str(int(grb.monthOfEndOfOverallTimeInterval)).zfill(2):2s}/01/{int(grb.yearOfEndOfOverallTimeInterval):4d}"
    var=grb.shortName
    PT=0
 
    for j in range(0,grb.Nj):
        for i in range(0,grb.Ni):
            PT+=1
            val=dat[0][j][i]
            lat=dat[1][j][i]
            lon=dat[2][j][i]  
            if PT not in data:
                data[PT]={}
            if adate not in data[PT]:
                data[PT][adate]={}
            if fdate not in data[PT][adate]:
                data[PT][adate][fdate]={}
            if nfct not in data[PT][adate][fdate]:
                data[PT][adate][fdate][nfct]={}
                
            if var not in data[PT][adate][fdate][nfct]:
                data[PT][adate][fdate][nfct][var]=val
            
            if PT not in latlons:
                latlons[PT]={}
                latlons[PT]["Lat"]=lat
                latlons[PT]["Lon"]=lon
    return data,latlons


def K2F(K):
       return (K - 273.15) * 9/5 + 32


def tTd2RH(t,td):
#  t,td should be in C
    rh=  100 * (math.exp((17.625 * td) / (243.04 + td))) / (math.exp((17.625 * t) / (243.04 + t)))
    return rh

def vpd(T,RH):
#    T should in in C
 #   tt=T-273.15
    tt=T
    es =0.6108*math.exp((17.27*tt)/(tt+237.3))
    if RH>1:
       val=100
    else:
        val=1
    ea=RH/val*es
    return es-ea
# Iterate over the messages in the file

#grbs = pygrib.open("ecmwf_51.grib")
grbs = pygrib.open("9035aad105e492216740525c32c4eeee.grib")


latlons={}
data={}
for grb in grbs:
    data,latlons=decodeGrib(grb,data,latlons)

       

In [21]:
print(grb.units,grb.name)
#calendar.monthrange(year, month)[1]
fdate = f"{str(int(grb.monthOfEndOfOverallTimeInterval)).zfill(2):2s}/01/{int(grb.yearOfEndOfOverallTimeInterval):4d}"
mon=int(grb.monthOfEndOfOverallTimeInterval)
yr = int(grb.yearOfEndOfOverallTimeInterval)

print(yr,mon,calendar.monthrange(yr, mon)[1])

m s**-1 Mean total precipitation rate
2025 7 31


In [22]:
data.keys()

dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40])

In [44]:
pwd

'/home/joe/work/Fire'

In [47]:
fout=open("Data/ecmwfTTdRhVpdUVP.csv","w")
fout.write("point,analysisDate,forecastDate,forecastInterval,t,td,rh,vpd,u,v,pcp\n")
for pt,dct in data.items():
    print(pt)
    for adate,dct1 in dct.items():
        for fdate,dct2 in dct1.items():
            for nfcts,dct3 in dct2.items():
                Td = dct3['2d']-273.15
                T = dct3['2t']-273.15
                U = dct3["10u"]
                V = dct3["10v"]
                PCP = dct3["tprate"]               
                RH=tTd2RH(T,Td)
                VPD=vpd(T,RH)
                data[pt][adate][fdate][nfcts]['rh']=RH
                data[pt][adate][fdate][nfcts]['vpd']=VPD
                fout.write(f"{pt},{adate},{fdate},{nfcts},{T:.1f},{Td:.1f},{RH:.1f},{VPD:.3f},{U:.4f},{V:.4f},{PCP}\n")
                
fout.close()        

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40


In [28]:
data.keys()

dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40])

In [24]:
fout=open("Data/ecmwf_51_latlons.csv","w")
fout.write("Point,Lat,Lon\n")
for pt,dct in latlons.items():
    fout.write(f"{pt},{dct['Lat']},{dct['Lon']}\n")
fout.close()

In [48]:
df=pd.read_csv("Data/ecmwfTTdRhVpdUVP.csv",index_col=False)

In [49]:
df.columns

Index(['point', 'analysisDate', 'forecastDate', 'forecastInterval', 't', 'td',
       'rh', 'vpd', 'u', 'v', 'pcp'],
      dtype='object')

In [50]:
df.shape

(23520, 11)

In [6]:
8*12+6

102

In [7]:
23520/102

230.58823529411765

In [8]:
8*12*6+30

606

In [9]:
23520/606

38.81188118811881

In [10]:
df.head()


point  analysisDate  forecastDate  forecastInterval  \
1 01/01/2017 01/01/2017      0         -10.8         -13.7              78.9   
             02/01/2017      1          -8.1         -11.6              76.0   
             03/01/2017      2          -3.7          -8.2              71.1   
             04/01/2017      3           2.1          -4.9              59.8   
             05/01/2017      4           9.7          -0.9              47.6   

                             t      td      rh           vpd  
1 01/01/2017 01/01/2017  0.057  1.2499  0.9278  1.748798e-08  
             02/01/2017  0.079  1.4632  0.6690  1.000410e-08  
             03/01/2017  0.134  1.4747  0.5108  1.393254e-08  
             04/01/2017  0.285  1.8091  0.4843  1.679983e-08  
             05/01/2017  0.628  1.6640  0.4686  1.745140e-08

In [11]:
df.index

MultiIndex([( 1, '01/01/2017', '01/01/2017'),
            ( 1, '01/01/2017', '02/01/2017'),
            ( 1, '01/01/2017', '03/01/2017'),
            ( 1, '01/01/2017', '04/01/2017'),
            ( 1, '01/01/2017', '05/01/2017'),
            ( 1, '01/01/2017', '06/01/2017'),
            ( 1, '02/01/2017', '02/01/2017'),
            ( 1, '02/01/2017', '03/01/2017'),
            ( 1, '02/01/2017', '04/01/2017'),
            ( 1, '02/01/2017', '05/01/2017'),
            ...
            (40, '01/01/2025', '03/01/2025'),
            (40, '01/01/2025', '04/01/2025'),
            (40, '01/01/2025', '05/01/2025'),
            (40, '01/01/2025', '06/01/2025'),
            (40, '02/01/2025', '02/01/2025'),
            (40, '02/01/2025', '03/01/2025'),
            (40, '02/01/2025', '04/01/2025'),
            (40, '02/01/2025', '05/01/2025'),
            (40, '02/01/2025', '06/01/2025'),
            (40, '02/01/2025', '07/01/2025')],
           length=23520)

In [12]:
df.reset_index(inplace=True)

In [13]:
df.head()

,level_0,level_1,level_2,point,analysisDate,forecastDate,forecastInterval,t,td,rh,vpd
0,1,01/01/2017,01/01/2017,0,-10.8,-13.7,78.9,0.057,1.2499,0.9278,1.748798e-08
1,1,01/01/2017,02/01/2017,1,-8.1,-11.6,76.0,0.079,1.4632,0.6690,1.000410e-08
2,1,01/01/2017,03/01/2017,2,-3.7,-8.2,71.1,0.134,1.4747,0.5108,1.393254e-08
3,1,01/01/2017,04/01/2017,3,2.1,-4.9,59.8,0.285,1.8091,0.4843,1.679983e-08
4,1,01/01/2017,05/01/2017,4,9.7,-0.9,47.6,0.628,1.6640,0.4686,1.745140e-08


In [14]:
df.groupby(["level_1"]).count()

,level_0,level_2,point,analysisDate,forecastDate,forecastInterval,t,td,rh,vpd
level_1,,,,,,,,,,
01/01/2017,240,240,240,240,240,240,240,240,240,240
01/01/2018,240,240,240,240,240,240,240,240,240,240
01/01/2019,240,240,240,240,240,240,240,240,240,240
01/01/2020,240,240,240,240,240,240,240,240,240,240
01/01/2021,240,240,240,240,240,240,240,240,240,240
...,...,...,...,...,...,...,...,...,...,...
12/01/2020,240,240,240,240,240,240,240,240,240,240
12/01/2021,240,240,240,240,240,240,240,240,240,240
12/01/2022,240,240,240,240,240,240,240,240,240,240


In [34]:
display(df["point"].value_counts())

point
1     588
2     588
3     588
4     588
5     588
6     588
7     588
8     588
9     588
10    588
11    588
12    588
13    588
14    588
15    588
16    588
17    588
18    588
19    588
20    588
21    588
22    588
23    588
24    588
25    588
26    588
27    588
28    588
29    588
30    588
31    588
32    588
33    588
34    588
35    588
36    588
37    588
38    588
39    588
40    588
Name: count, dtype: int64

In [38]:
df.columns

Index(['point', 'analysisDate', 'forecastDate', 'forecastInterval', 't', 'td',
       'rh', 'vpd'],
      dtype='object')

In [40]:
df["analysisDate"].value_counts()

analysisDate
01/01/2017    240
02/01/2017    240
03/01/2017    240
04/01/2017    240
05/01/2017    240
             ... 
10/01/2024    240
11/01/2024    240
12/01/2024    240
01/01/2025    240
02/01/2025    240
Name: count, Length: 98, dtype: int64

In [16]:
ll = pd.read_csv("Data/ecmwf_51_latlons.csv")

In [17]:
ll.head()

,Point,Lat,Lon
0,1,41.0,-109.0
1,2,41.0,-108.0
2,3,41.0,-107.0
3,4,41.0,-106.0
4,5,41.0,-105.0


In [18]:
ll.head(100)

,Point,Lat,Lon
0,1,41.0,-109.0
1,2,41.0,-108.0
2,3,41.0,-107.0
3,4,41.0,-106.0
4,5,41.0,-105.0
5,6,41.0,-104.0
6,7,41.0,-103.0
7,8,41.0,-102.0
8,9,40.0,-109.0
9,10,40.0,-108.0
